# 1.1 Understanding Statistics

**Goal:** distinguish describing observed data from making claims about a wider population, and connect both to AI/ML decisions.

**Prerequisites:** arithmetic, percentages, and summation notation from Module 0. **Runtime:** Python 3.10+, standard library only. Saved outputs are included. Use **Restart Kernel and Run All** to reproduce them. Every example is synthetic and illustrative.

## Contents
1. What is statistics?
2. Why statistics matters in AI and ML
3. Descriptive statistics
4. Inferential statistics
5. Population, sample, and statistical population
6. Parameters and statistics
7. Census versus sampling
8. ML case study, practice, and solutions

## 1. What is Statistics?

**Statistics** is the discipline of collecting, organizing, summarizing, analyzing, and interpreting data while accounting for variability and uncertainty. A **statistic** (singular) is also a numerical summary computed from a sample; distinguish the discipline from a particular summary.

An observation records values for an **observational unit**: a person, transaction, device, image, or other entity. A variable is a characteristic recorded for that unit. A dataset may contain many rows per unit: repeated visits from one patient are not automatically independent observations.

A useful workflow is:

1. State the question and the decision it supports.
2. Define the target population and the observational unit.
3. Collect or sample data using a documented process.
4. Check quality and summarize the observed data.
5. Use an appropriate model or inferential method, checking assumptions.
6. Communicate what the evidence supports and what remains uncertain.

**Worked example:** “How long do customers wait?” is incomplete. “What is the mean weekday checkout wait, in minutes, among all customers at Store A during November?” specifies the variable, population, place, and time. Sampling only the morning shift answers a narrower question unless a defensible design accounts for other shifts.

## 2. Why Statistics is important in AI and ML

| ML task | Statistical question |
|---|---|
| Understand a dataset | Which values, groups, outliers, and missing entries occur? |
| Select features | Are associations plausible, stable, and available at prediction time? |
| Preprocess data | What should be scaled, encoded, or imputed, and using which training data? |
| Fit a model | Which parameters minimize a loss or explain observed variation? |
| Evaluate performance | Does the test set represent intended deployment cases? How uncertain is its score? |
| Monitor deployment | Have inputs, labels, or error patterns changed over time? |

A high test accuracy is evidence about performance on the test distribution, not a guarantee about every future user or a proof of causality. Data quality and sampling design matter alongside algorithms.

## 3. Descriptive Statistics

Descriptive statistics organize and summarize the data actually observed. They include counts, proportions, means, medians, spread, and visualizations. A summary can describe a sample or a complete population without generalizing beyond it.

For values $x_1,\ldots,x_n$, the mean is $\bar x=\frac1n\sum_i x_i$. The median is the middle sorted value (or the mean of the two middle values when $n$ is even). The range is maximum minus minimum. The sample variance is $s^2=\sum_i(x_i-\bar x)^2/(n-1)$ for $n>1$.

**Worked example:** waits of 2, 3, 3, 4, and 18 minutes have mean 6, median 3, and range 16. The long wait pulls the mean upward. Neither the mean alone nor the median alone tells the full story. Module 2 develops these summaries in depth.

In [1]:
import statistics
from collections import Counter

waits = [2, 3, 3, 4, 18]
print("Observed waits:", waits)
print("Count:", len(waits), "Mean:", statistics.mean(waits))
print("Median:", statistics.median(waits), "Range:", max(waits)-min(waits))
print("Sample variance:", statistics.variance(waits))
print("Frequencies:", dict(sorted(Counter(waits).items())))
assert statistics.mean(waits) == 6
assert statistics.median(waits) == 3

Observed waits: [2, 3, 3, 4, 18]
Count: 5 Mean: 6
Median: 3 Range: 16
Sample variance: 45.5
Frequencies: {2: 1, 3: 2, 4: 1, 18: 1}


## 4. Inferential Statistics

Inferential statistics use sample data and assumptions about collection or a statistical model to learn about a wider population or generating process. Examples include estimating a population mean, constructing confidence intervals, and testing hypotheses.

A useful distinction is **sampling variability** versus **bias**. Different random samples give different answers even under a sound design. Bias is a systematic tendency to miss the intended quantity, for example when a survey excludes a group. A larger sample can reduce random error while retaining selection bias.

**Worked example:** the mean wait of 6 minutes is a descriptive fact for the five recorded visits. Claiming that the mean for all visits is about 6 minutes is an inference that needs justification about how those visits were selected.

The simulation below repeatedly samples from a completely known toy population. We can see the true mean here; it is usually unknown in real inference. The spread of sample means illustrates a **sampling distribution**. It is not the distribution of individual values, and its spread is not the same as the population standard deviation. Confidence intervals and formal tests come later in the syllabus.

In [2]:
import random
rng = random.Random(42)
population = list(range(1, 101))
true_mean = statistics.mean(population)
for sample_size in [5, 25]:
    sample_means = [statistics.mean(rng.sample(population, sample_size)) for _ in range(2000)]
    print(f"n={sample_size}: average sample mean={statistics.mean(sample_means):.3f}, "
          f"SD of sample means={statistics.stdev(sample_means):.3f}")
print("Known population mean:", true_mean)
# Many repetitions illustrate typical behavior; they are not a proof.

n=5: average sample mean=51.270, SD of sample means=12.647
n=25: average sample mean=50.509, SD of sample means=5.218
Known population mean: 50.5


## 5. Population and Sample; Statistical population

A **statistical population** is the complete set of units or measurements relevant to a question. It need not be a set of people: it can be all production batches, all eligible images, or a modeled stream of future requests. State eligibility, location, and time boundaries explicitly.

A **sample** is the subset observed or selected to study that population. A **sampling frame** is the operational list or mechanism from which units can actually be selected. The **target population** is the group the question concerns; the frame and observed sample may cover less than this target.

**Worked example:**

- Target population: all active app users in India during November.
- Unit: one user, not one session.
- Frame: accounts with a usable contact address.
- Selected sample: randomly chosen accounts from that frame.
- Observed sample: selected accounts whose users actually respond.

Users without contact details and selected nonrespondents are absent from the observed data. Random selection from the frame alone does not fix incomplete frame coverage or nonresponse. If a highly active user produces 100 rows, treating those rows as 100 users changes the unit and can exaggerate effective information.

**Representativeness** is about how well the data support the intended question; it cannot be established simply by calling a sample “large.” A probability sample uses known selection probabilities and may require weights when those probabilities differ.

## 6. Parameter and Statistic

A **parameter** is a fixed, usually unknown numerical characteristic of the population or model. A **statistic** is computed from sample data without depending on an unknown parameter. An **estimator** is the rule used to estimate a parameter; its realized output is an **estimate**.

| Population parameter | Sample statistic/estimator |
|---|---|
| Mean $\mu$ | Sample mean $\bar x$ |
| Variance $\sigma^2$ | Sample variance $s^2$ |
| Proportion $p$ | Sample proportion $\hat p=k/n$ |

For a finite population of size $N$, $\mu=\sum_{i=1}^N x_i/N$ and $\sigma^2=\sum_{i=1}^N(x_i-\mu)^2/N$. The sample formula with $n-1$ serves a different estimation purpose from describing a complete finite population with denominator $N$.

**Worked example:** if 18 of 20 sampled requests succeed, $\hat p=18/20=0.9$. The success proportion of all target requests is the unknown parameter $p$. Another sample would generally give a different $\hat p$.

In [3]:
rng = random.Random(7)
all_outcomes = [1] * 80 + [0] * 20
population_p = statistics.mean(all_outcomes)
sample = rng.sample(all_outcomes, 20)
sample_p = statistics.mean(sample)
print("Toy population parameter p:", population_p)
print("Sample estimate p-hat:", sample_p)
print("Population variance:", statistics.pvariance(all_outcomes))
print("Sample variance estimator:", statistics.variance(sample))

Toy population parameter p: 0.8
Sample estimate p-hat: 0.95
Population variance: 0.16
Sample variance estimator: 0.05


## 7. Census vs. Sampling

A **census** attempts to collect data from every unit in the defined population. **Sampling** collects a subset. “Census” describes intended coverage; missing responses and recording errors can still prevent a perfect count.

| Consideration | Census | Sample |
|---|---|---|
| Coverage intended | Every eligible unit | Selected subset |
| Cost/time | Often higher | Often lower |
| Sampling error | None for a complete, accurately observed finite population | Present under random sampling |
| Other errors | Coverage, nonresponse, measurement, processing errors remain possible | Same errors remain possible |
| Practical use | Small inventories, complete operational records | Large populations, destructive tests, rapid studies |

A complete log of recorded events may still omit failed logging events or people who never used the product. Even an accurate census of today's users does not directly answer questions about tomorrow's users.

**Worked example:** inspecting every unit in a 40-item stockroom can be practical. Measuring product lifetime by destroying every item would leave no stock to sell; sampling is necessary. Careful measurement on a smaller sample can sometimes be more useful than poor measurement in a census.

## 8. AI/ML case study: what does accuracy describe?

A classifier predicts 90 of 100 held-out labeled examples correctly. Its observed accuracy is 0.90. Generalizing to future cases requires a suitable test design, reliable labels, and attention to user overlap, temporal ordering, and deployment changes. Overall accuracy can conceal subgroup failures.

In [4]:
groups = {"A": {"correct": 81, "total": 90},
          "B": {"correct": 9, "total": 10}}
for name, counts in groups.items():
    print(name, "accuracy:", counts["correct"] / counts["total"], "n:", counts["total"])
correct = sum(g["correct"] for g in groups.values())
total = sum(g["total"] for g in groups.values())
print("Observed overall accuracy:", correct / total)
print("Equal point estimates do not imply equal uncertainty: group B has fewer observations.")
assert correct / total == 0.9

A accuracy: 0.9 n: 90
B accuracy: 0.9 n: 10
Observed overall accuracy: 0.9
Equal point estimates do not imply equal uncertainty: group B has fewer observations.


## Practice

1. Define the population, unit, and variable for a study of average delivery time in a city next month.
2. “The median of these 300 orders is 24 minutes.” Descriptive or inferential? What changes if the claim is about all next-month orders?
3. A random sample contains 42 satisfied customers out of 60. Identify the statistic and parameter.
4. A survey of all employees has a 60% response rate. Is its observed dataset a complete census?
5. Why might one million sessions provide less information than one million independent users?
6. Explain why increasing the size of a survey restricted to premium users may not estimate satisfaction of all users well.
7. A test set has 99% accuracy and contains almost no examples from a deployment region. What can and cannot be concluded?

## Solutions

1. Population: all eligible deliveries in the defined city and next-month window. Unit: one delivery. Variable: elapsed minutes between explicitly defined start/end events. Specify exclusions such as canceled orders.
2. The first is descriptive. A claim about future orders is inferential and needs a sampling/model justification and uncertainty assessment.
3. Statistic $\hat p=42/60=0.70$; parameter $p$ is the satisfaction proportion in the defined target customer population.
4. It is an attempted census with nonresponse, not a complete observed census. Respondents may differ systematically from nonrespondents.
5. Repeated sessions from the same user may be correlated; row count does not equal the count of independent units.
6. It retains coverage/selection bias if nonpremium users differ. More observations do not automatically remove systematic bias.
7. The model achieved 99% on the observed test set. That result gives weak evidence about the underrepresented region and cannot guarantee its deployment accuracy.

**Readiness check:** state the target population before calculating a summary, separate parameters from statistics, and explain the assumptions behind a generalization.

**Next:** [1.2 Types of Data](2_Types-of-Data.ipynb).